# Simulation of Markov Chains with Quadratic Energy Kernels

we simulate synthetic data from a Markov process in $\mathbb{R}^{10}$.  
We model the transition kernel using a quadratic energy function:
$$
E(y, x) = \frac{1}{2}(y - A x - b)^\top \Sigma^{-1} (y - A x - b),
$$

which corresponds to a Gaussian conditional distribution
$$
y \mid x \sim \mathcal{N}(A x + b, \Sigma).
$$
We define two different kernels:
- **Pre-change kernel $P$** with parameters $(A_0, b_0, \Sigma_0)$  
- **Post-change kernel $Q$** with parameters $(A_1, b_1, \Sigma_1)$  

At a chosen change point $\nu$, the process switches from $P$ to $Q$.  
This allows us to generate a sample path with a known change point, which we will later use to test our Hyvärinen score-based change detection method.


In [4]:
import numpy as np
import torch
from tqdm import tqdm

In [5]:
def sample_markov_chain(n_steps, A, b, Sigma, x0=None, seed=None, show_progress=False):
    """
    Generate a sample path from a Gaussian Markov chain:
        X_{t+1} | X_t ~ N(A X_t + b, Sigma)

    Parameters
    ----------
    n_steps : int
        Number of transitions (path will have length n_steps+1).
    A : np.ndarray (d x d)
        Linear transformation matrix.
    b : np.ndarray (d,)
        Bias vector.
    Sigma : np.ndarray (d x d)
        Covariance matrix (positive definite).
    x0 : np.ndarray (d,), optional
        Initial state. Defaults to zero vector.
    seed : int, optional
        Random seed for reproducibility.
    show_progress : bool, optional
        If True, show tqdm progress bar.

    Returns
    -------
    X : np.ndarray of shape (n_steps+1, d)
        The simulated Markov chain sample path.
    """
    rng = np.random.default_rng(seed)
    d = A.shape[0]

    # Initialize
    if x0 is None:
        x = np.zeros(d)
    else:
        x = np.array(x0)

    X = [x]
    iterator = range(n_steps)
    if show_progress:
        iterator = tqdm(iterator, desc="Simulating Markov chain")

    for _ in iterator:
        noise = rng.multivariate_normal(mean=np.zeros(d), cov=Sigma)
        x = A @ x + b + noise
        X.append(x)

    return np.array(X)


### Pre-change Markov Kernel $P$

We define a Gaussian Markov process $ \{X_n\} \subset \mathbb{R}^{10} $ with transition dynamics:
$$
X_{n+1} = A_0 X_n + b_0 + \varepsilon_n, \quad \varepsilon_n \sim \mathcal{N}(0, \Sigma_0)
$$

The parameters for the **pre-change kernel $P$** are:

- $A_0 = 0.8 \cdot I_{10} \in \mathbb{R}^{10 \times 10} $  
- $b_0 = \mathbf{0} \in \mathbb{R}^{10} $
- $ \Sigma_0 = 0.1 \cdot I_{10} \in \mathbb{R}^{10 \times 10} $

This defines a **linear-Gaussian transition kernel** where $X_{n+1} \mid X_n \sim \mathcal{N}(A_0 X_n + b_0, \Sigma_0)$.



### Sampling Process

Given an initial point $ X_0 = 0 $, we simulate the Markov chain recursively using the formula above. The noise $ \varepsilon_n $ is i.i.d. Gaussian.


### Stationary Distribution

This Markov chain is geometrically ergodic and converges to its stationary distribution:

- Mean: $ \mu_\infty = (I - A_0)^{-1} b_0 = \mathbf{0} $
- Covariance:$ \Sigma_\infty = A \Sigma_\infty A^\top + \Sigma $.  Here, we have  
$
\Sigma_\infty = a^2 \Sigma_\infty + \Sigma
\Rightarrow \Sigma_\infty (1 - a^2) = \Sigma
\Rightarrow \Sigma_\infty = \frac{\Sigma}{1 - a^2}
$


### Mixing Time

The covariance error decays as:

$$
\| \Sigma_n - \Sigma_\infty \| \le \rho(A_0)^{2n} \cdot \| \Sigma_0 - \Sigma_\infty \|
$$

For a tolerance of $ \varepsilon = 10^{-4} $, we solve:

$$
0.8^{2n}   \le 10^{-4} \quad \Rightarrow \quad n \ge 21
$$

Thus, we discard the first **21 samples** as burn-in and treat all subsequent samples as drawn from the stationary distribution.


In [6]:
# --- Example usage ---
d = 10
seed = 42
A_P = 0.8 * np.eye(d)
b_P = np.random.rand(d)
Sigma_P = 0.1 * np.eye(d)

n_pre = 100000

X_pre = sample_markov_chain(
    n_steps=n_pre,
    A=A_P,
    b=b_P,
    Sigma=Sigma_P,
    seed=seed,
    show_progress=True  # <--- tqdm on
)

print("Full path shape:", X_pre.shape)
torch.save(torch.from_numpy(X_pre), "markov_path_A_0.8.pt")

Simulating Markov chain: 100%|███████████████████████████████████████| 100000/100000 [00:32<00:00, 3084.04it/s]


Full path shape: (100001, 10)


In [ ]:

# --- Dimension ---
d = 10

# --- Pre-change kernel P ---
A_P = 0.8 * np.eye(d)                 # contraction
b_P = np.zeros(d)                     # no shift
Sigma_P = 0.1 * np.eye(d)             # small noise

# --- Post-change kernel Q ---
A_Q = 0.6 * np.eye(d)                 # different contraction
b_Q = 0.3 * np.ones(d)                # mean shift
Sigma_Q = 0.3 * np.eye(d)             # larger noise

# --- Simulation parameters ---
n_pre = 30000    # steps before change
n_post = 600    # steps after change
seed = 42       # reproducibility

# --- Simulate pre-change path ---
X_pre = sample_markov_chain(
    n_steps=n_pre,
    A=A_P,
    b=b_P,
    Sigma=Sigma_P,
    x0=None,
    seed=seed
)

# --- Simulate post-change path, starting from last pre-change state ---
# X_post = sample_markov_chain(
#     n_steps=n_post,
#     A=A_Q,
#     b=b_Q,
#     Sigma=Sigma_Q,
#     x0=X_pre[-1],
#     seed=seed+1
# )

# # --- Concatenate to form full path ---
# X = np.vstack([X_pre])

print("Full path shape:", X_pre.shape)   # (n_pre+n_post+1, d)
# print("Change point at index:", n_pre)

torch.save(torch.from_numpy(X), "markov_path_P.pt")

In [5]:
X_loaded = torch.load("markov_path_P.pt")
print(type(X_loaded))   # should be <class 'torch.Tensor'>
print(X_loaded.shape)   # (n_steps+1, d)
print(X_loaded.dtype)   # e.g. torch.float32
print(X_loaded[:5])     # first 5 rows
print(X_loaded[-5:])    # last 5 rows


<class 'torch.Tensor'>
torch.Size([100001, 10])
torch.float64
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [ 0.0964, -0.3289,  0.2373,  0.2974, -0.6170, -0.4118,  0.0404, -0.1000,
         -0.0053, -0.2698],
        [ 0.3552, -0.0171,  0.2107,  0.5944, -0.3457, -0.6012,  0.1490, -0.3832,
          0.2735, -0.2316],
        [ 0.2257, -0.2290,  0.5552,  0.4267, -0.4120, -0.5923,  0.2875, -0.1910,
          0.3493, -0.0490],
        [ 0.8578, -0.3118,  0.2822,  0.0840, -0.1348, -0.1168,  0.1940, -0.4185,
          0.0188,  0.1665]], dtype=torch.float64)
tensor([[ 0.0680, -0.0764,  1.1175, -0.6576, -0.4217,  0.0750,  0.0533,  0.3381,
          0.0701,  1.2047],
        [-0.1802,  0.4413,  1.1645, -0.3845, -0.2552,  0.7272,  0.1435,  0.2231,
          0.2140,  0.7392],
        [ 0.0440,  0.1856,  1.1205, -0.9143,  0.2029,  0.3058,  0.1072,  0.8982,
          0.0506,  0.0566],
        [ 0.3313, -0.3131,  0.9679, -0.9224,